Q1. Install MLflow

In [2]:
#INSTALLING LIBRARIES - mlflow scikit-learn pandas seaborn hyperopt xgboost fastparquet boto3
import sys

!{sys.executable} -m pip install --no-user mlflow scikit-learn pandas seaborn hyperopt xgboost fastparquet boto3

  Using cached mlflow-3.18.0-py3-none-any.whl.metadata (50 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached pandas-3.1.0-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached hyperopt-0.3.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached xgboost-3.4.1-py3-none-manylinux_2_28_x86_64.whl.metadata (2.0 kB)
  Using cached fastparquet-2026.9.0-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (4.8 kB)
  Using cached boto3-1.43.123-py3-none-any.whl.metadata (6.6 kB)
  Using cached mlflow_skinny-3.18.0-py3-none-any.whl.metadata (51 kB)
  Using cached mlflow_tracing-3.18.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached docker-7.2.0-py3-none-a

In [3]:
# CHECK LIBRARIES ARE SUCCESSFULLY INSTALLED
import mlflow
import sklearn
import pandas
import seaborn
import hyperopt
import xgboost
import fastparquet
import boto3

print("All packages installed successfully!")

/srv/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All packages installed successfully!


In [4]:
# CREATE DATA FOLDER AND OUTPUT FOLDER
import os

DATA_FOLDER = os.path.abspath("taxi_data")
OUTPUT_FOLDER = os.path.abspath("output")

os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Data folder:", DATA_FOLDER)
print("Output folder:", OUTPUT_FOLDER)

Data folder: /home/test/taxi_data
Output folder: /home/test/output


In [6]:
# DOWNLOAD GREEN TAXI TRIP DATA 2024 - JAN, FEB, MARCH AND PUSH TO DATA_FOLDER = taxi_data
import requests
from pathlib import Path

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

months = ["01", "02", "03"]

for month in months:
    filename = f"green_tripdata_2023-{month}.parquet"
    url = base_url + filename
    destination = Path(DATA_FOLDER) / filename

    print(f"Downloading {filename}...")

    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()

    with open(destination, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(f"Downloaded: {destination}")

print("All three datasets downloaded successfully!")

Downloaded: /home/test/taxi_data/green_tripdata_2023-01.parquet
Downloaded: /home/test/taxi_data/green_tripdata_2023-02.parquet
Downloaded: /home/test/taxi_data/green_tripdata_2023-03.parquet
All three datasets downloaded successfully!


In [14]:
import os
import pickle
import pandas as pd
from sklearn.feature_extraction import DictVectorizer


def dump_pickle(obj, filename: str):
    with open(filename, "wb") as f_out:
        return pickle.dump(obj, f_out)


def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


def preprocess(df: pd.DataFrame, dv: DictVectorizer, fit_dv: bool = False):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    if fit_dv:
        X = dv.fit_transform(dicts)
    else:
        X = dv.transform(dicts)
    return X, dv


def run_data_prep(raw_data_path: str, dest_path: str, dataset: str = "green"):
    # Load parquet files
    df_train = read_dataframe(
        os.path.join(raw_data_path, f"{dataset}_tripdata_2023-01.parquet")
    )
    df_val = read_dataframe(
        os.path.join(raw_data_path, f"{dataset}_tripdata_2023-02.parquet")
    )
    df_test = read_dataframe(
        os.path.join(raw_data_path, f"{dataset}_tripdata_2023-03.parquet")
    )

    # Extract the target
    target = 'duration'
    y_train = df_train[target].values
    y_val = df_val[target].values
    y_test = df_test[target].values

    # Fit the DictVectorizer and preprocess data
    dv = DictVectorizer()
    X_train, dv = preprocess(df_train, dv, fit_dv=True)
    X_val, _ = preprocess(df_val, dv, fit_dv=False)
    X_test, _ = preprocess(df_test, dv, fit_dv=False)

    # Create dest_path folder unless it already exists
    os.makedirs(dest_path, exist_ok=True)

    # Save DictVectorizer and datasets
    dump_pickle(dv, os.path.join(dest_path, "dv.pkl"))
    dump_pickle((X_train, y_train), os.path.join(dest_path, "train.pkl"))
    dump_pickle((X_val, y_val), os.path.join(dest_path, "val.pkl"))
    dump_pickle((X_test, y_test), os.path.join(dest_path, "test.pkl"))




In [16]:
run_data_prep('./taxi_data','./output')

Q2 How many files were saved to OUTPUT_FOLDER?
- 4 files
1. dv.pkl
2. test.pkl
3. train.pkl
4. val.pkl


Q3. Train a model with autolog
# task is to modify the script to enable autologging with MLflow
# execute the script and then launch the MLflow UI to check that the experiment run was properly tracked.

In [32]:
import os
import pickle
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)

def run_train(data_path: str):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("random-forest-experiment")
    # Enable sklearn autologging with safe serialization
    mlflow.sklearn.autolog(
        log_models=True,
        serialization_format="skops"
    )

    with mlflow.start_run():
        print("start")
        X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
        X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        print("stop")
        return {rmse}

In [33]:
run_train('./output')

start


2026/11/04 13:18:39 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the

stop
🏃 View run tasteful-bass-199 at: http://127.0.0.1:5000/#/experiments/1/runs/30d2313a7bc141c6ad428c9bb97a509c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


{np.float64(5.431151718191326)}

Q. What is the value of the min_samples_split parameter:
2


In [ ]:
Q. select a SQLite db for the backend store and a folder called artifacts for the artifacts store.
# ran this command for the same
mlflow server \
    --backend-store-uri sqlite:///mlflow.db \
    --default-artifact-root ./artifacts \
    --host 0.0.0.0 \
    --port 5000

Q In addition to backend-store-uri, what else do you need to pass to properly configure the server?
default-artifact-root

In [37]:
import os
import pickle
import click
import mlflow
import numpy as np
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("random-forest-hyperopt")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)

def run_optimization(data_path: str, num_trials: int):

    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    def objective(params):
        with mlflow.start_run():
            # Log hyperparameters
            mlflow.log_params(params)
    
            rf = RandomForestRegressor(**params)
            rf.fit(X_train, y_train)
    
            y_pred = rf.predict(X_val)
    
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
            # Log validation RMSE
            mlflow.log_metric("rmse", rmse)

        return {'loss': rmse, 'status': STATUS_OK}
 
    search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

    rstate = np.random.default_rng(42)  # for reproducible results
    fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=num_trials,
        trials=Trials(),
        rstate=rstate
    )

In [39]:
run_optimization(
        data_path="./output",
        num_trials=50
    )

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

2026/11/04 13:49:06 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the

🏃 View run dashing-snipe-865 at: http://127.0.0.1:5000/#/experiments/2/runs/5597a01622e5414aab4f0ac3f37cb6ac

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

  2%|▏         | 1/50 [00:09<07:49,  9.58s/trial, best loss: 5.370086069268862]

2026/11/04 13:49:09 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the

🏃 View run victorious-steed-604 at: http://127.0.0.1:5000/#/experiments/2/runs/623a1dcc20ad4333bc7ad5a0c34dcbbb

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

  4%|▍         | 2/50 [00:12<04:35,  5.74s/trial, best loss: 5.370086069268862]

2026/11/04 13:49:11 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the

... (91 outputs left out)

2026/11/04 13:54:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the

🏃 View run mysterious-ant-461 at: http://127.0.0.1:5000/#/experiments/2/runs/53431218bdf4467e98dd1ba9e092f212

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                    

 98%|█████████▊| 49/50 [05:24<00:05,  5.12s/trial, best loss: 5.320408422837524]

2026/11/04 13:54:26 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the

🏃 View run casual-shrew-134 at: http://127.0.0.1:5000/#/experiments/2/runs/85a8af8058774714b7977dac3a83c65e

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                    

100%|██████████| 50/50 [05:29<00:00,  6.59s/trial, best loss: 5.320408422837524]


In [40]:
# list down all the 50 runs to check the max depth and metrics rmse
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

print("Tracking URI:", mlflow.get_tracking_uri())

experiment = mlflow.get_experiment_by_name(
    "random-forest-hyperopt"
)

print("Experiment:", experiment)

if experiment:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )
    print(runs[["params.max_depth", "metrics.rmse"]])

Tracking URI: http://127.0.0.1:5000
Experiment: <Experiment: artifact_location='/home/test/artifacts/2', creation_time=1793798761931, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1793798761931, lifecycle_stage='active', name='random-forest-hyperopt', tags={}, trace_location=None, workspace='default'>
   params.max_depth  metrics.rmse
0                15      5.348106
1                 5      5.543964
2                12      5.410422
3                11      5.389385
4                20      5.362385
5                 9      5.425467
6                19      5.368692
7                18      5.327806
8                 3      5.772942
9                15      5.352378
10               13      5.401477
11               17      5.340555
12               15      5.351053
13               19      5.325878
14               17      5.378654
15                1      7.237690
16               14      5.355690
17               11      5.391279
18               16 

In [41]:
#Find the best-performing run
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

best_run = runs.loc[runs["metrics.rmse"].idxmin()]

print("Best RMSE:", best_run["metrics.rmse"])
print("Best parameters:")
print(best_run.filter(like="params."))

Best RMSE: 5.320408422837524
Best parameters:
params.criterion                   squared_error
params.random_state                           42
params.n_estimators                           31
params.max_depth                              20
params.max_features                          1.0
params.min_samples_leaf                        2
params.max_samples                          None
params.bootstrap                            True
params.min_samples_split                       4
params.ccp_alpha                             0.0
params.max_leaf_nodes                       None
params.monotonic_cst                        None
params.n_jobs                               None
params.verbose                                 0
params.min_weight_fraction_leaf              0.0
params.oob_score                           False
params.min_impurity_decrease                 0.0
params.warm_start                          False
Name: 24, dtype: object


What's the best validation RMSE that you got?
Best RMSE: 5.320408422837524

In [46]:

import os
import pickle
import mlflow

from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

HPO_EXPERIMENT_NAME = "random-forest-hyperopt"
EXPERIMENT_NAME = "random-forest-best-models"

RF_PARAMS = [
    'max_depth',
    'n_estimators',
    'min_samples_split',
    'min_samples_leaf',
    'random_state'
]

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.sklearn.autolog()


def load_pickle(filename):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


def train_and_log_model(data_path, params):
    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))
    X_test, y_test = load_pickle(os.path.join(data_path, "test.pkl"))

    with mlflow.start_run():
        new_params = {}

        for param in RF_PARAMS:
            new_params[param] = int(params[param])

        rf = RandomForestRegressor(**new_params)
        rf.fit(X_train, y_train)

        val_rmse = np.sqrt(mean_squared_error(y_val, rf.predict(X_val)))
        mlflow.log_metric("val_rmse", val_rmse)

        test_rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
        mlflow.log_metric("test_rmse", test_rmse)


# @click.command()
# @click.option(
#     "--data_path",
#     default="./output",
#     help="Location where the processed NYC taxi trip data was saved"
# )
# @click.option(
#     "--top_n",
#     default=5,
#     type=int,
#     help="Number of top models that need to be evaluated"
# )
def run_register_model(data_path: str, top_n: int):

    client = MlflowClient()

    # Retrieve top N models from hyperparameter optimization
    experiment = client.get_experiment_by_name(HPO_EXPERIMENT_NAME)

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=top_n,
        order_by=["metrics.rmse ASC"]
    )

    # Train and evaluate the selected models
    for run in runs:
        train_and_log_model(
            data_path=data_path,
            params=run.data.params
        )

    # Select the model with the lowest test RMSE
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

    best_run = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
        order_by=["metrics.test_rmse ASC"],
        max_results=1
    )[0]

    print("Best test RMSE:", best_run.data.metrics["test_rmse"])
    print("Best run ID:", best_run.info.run_id)

    # Register the best model
    model_uri = f"runs:/{best_run.info.run_id}/model"

    mlflow.register_model(
        model_uri=model_uri,
        name="random-forest-best-model"
    )


In [47]:
run_register_model('./output',5)


2026/11/04 14:17:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run merciful-goose-617 at: http://127.0.0.1:5000/#/experiments/3/runs/40bc8b82eda044f48235368f35df0259
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/11/04 14:17:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run popular-stoat-77 at: http://127.0.0.1:5000/#/experiments/3/runs/87e1a7da4b624abab32b31512bfbb945
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/11/04 14:17:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run bold-cod-651 at: http://127.0.0.1:5000/#/experiments/3/runs/01bcefd0d9764db0b1bb3d32f395433d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/11/04 14:17:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run amusing-shrike-429 at: http://127.0.0.1:5000/#/experiments/3/runs/6c6459df59f344d7b9d07eb980d0a2b1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/11/04 14:18:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run silent-stork-814 at: http://127.0.0.1:5000/#/experiments/3/runs/f6779960795d4d93949c3968a473870e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Best test RMSE: 5.55368157057841
Best run ID: 01bcefd0d9764db0b1bb3d32f395433d


Successfully registered model 'random-forest-best-model'.
2026/11/04 14:18:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 01bcefd0d9764db0b1bb3d32f395433d has no artifacts at artifact path 'model', registering model based on models:/m-9ef17c67695d444fa8e1fa462234d438 instead
2026/11/04 14:18:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random-forest-best-model, version 1
Created version '1' of model 'random-forest-best-model'.


In [ ]:
What is the test RMSE of the best model?
5.53